<a href="https://colab.research.google.com/github/jc13605-0721/ECE_9533_LLM4ChipDesign/blob/main/HW1_ChipChat/chipchat_extension.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Getting set up

## Setting up the notebook

In [11]:
### Installing dependencies
!pip install openai

!apt-get update
!apt-get install -y iverilog

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
iverilog is already the newest version (11.0-1.1).
0 upgraded, 0 newly installed, 0 to remove 

## Use LLM to generate Verilog code from natural language description

### Example 8: Traffic Light

In [12]:
!mkdir -p traffic_light/
!cd traffic_light/ && curl -O https://raw.githubusercontent.com/FCHXWH823/LLM4ChipDesign/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/traffic_light_tb.v

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  2018  100  2018    0     0   9252      0 --:--:-- --:--:-- --:--:--  9299


In [13]:
verilog_generation_prompt = '''
I already have a working traffic light FSM, and I would like to extend it slightly.

Please generate a synthesizable Verilog-2001 module named traffic_light_fsm with the following interface:

- input  wire clk
- input  wire reset_n   (active-low reset)
- input  wire enable
- output reg  red
- output reg  yellow
- output reg  green
- output reg  [1:0] state_code

The behavior of the traffic light should remain the same as before:
- After reset, the FSM starts in the RED state.
- When enable = 1:
  - Stay in RED for 32 clock cycles, then transition to GREEN.
  - Stay in GREEN for 20 clock cycles, then transition to YELLOW.
  - Stay in YELLOW for 7 clock cycles, then transition back to RED.
- When enable = 0, the FSM should hold its current state and counter.
- Only one of red, yellow, or green should be asserted at any time.

Extension requirement:
- Add the 2-bit output state_code to explicitly encode the current FSM state:
  - 2'b00 = RED
  - 2'b01 = GREEN
  - 2'b10 = YELLOW
- The value of state_code should always be consistent with the red/yellow/green outputs.

Implementation notes:
- A simple FSM with a state register and a counter is sufficient.
- Use plain Verilog-2001 style (no typedef, enum, logic, always_comb, or SystemVerilog-only syntax).
- Use a synchronous always block for state and counter updates, and a combinational block for outputs.
- No delays, no initial blocks.

Please output only the complete module code.


'''

In [14]:
from openai import OpenAI
from google.colab import userdata
import os

# Load API key from Colab Secrets
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# Create OpenAI client
client = OpenAI()

completion = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": verilog_generation_prompt}],
    max_tokens=1024,
)

print(completion.choices[0].message.content)


```verilog
module traffic_light_fsm (
    input wire clk,
    input wire reset_n,
    input wire enable,
    output reg red,
    output reg yellow,
    output reg green,
    output reg [1:0] state_code
);

    // State encoding
    localparam RED    = 2'b00;
    localparam GREEN  = 2'b01;
    localparam YELLOW = 2'b10;

    // Timer values
    localparam RED_TIME    = 32;
    localparam GREEN_TIME  = 20;
    localparam YELLOW_TIME = 7;

    reg [1:0] state;
    reg [4:0] counter; // Enough for counting up to 32

    // State and counter updates
    always @(posedge clk or negedge reset_n) begin
        if (!reset_n) begin
            state <= RED;
            counter <= 0;
        end else if (enable) begin
            case (state)
                RED: begin
                    if (counter == RED_TIME - 1) begin
                        state <= GREEN;
                        counter <= 0;
                    end else
                        counter <= counter + 1;
                end



In [19]:
output_verilog_code = '''
module traffic_light_fsm (
    input wire clk,
    input wire reset_n,
    input wire enable,
    output reg red,
    output reg yellow,
    output reg green,
    output reg [1:0] state_code
);

    // State encoding
    localparam RED    = 2'b00;
    localparam GREEN  = 2'b01;
    localparam YELLOW = 2'b10;

    // Timer values
    localparam RED_TIME    = 32;
    localparam GREEN_TIME  = 20;
    localparam YELLOW_TIME = 7;

    reg [1:0] state;
    reg [4:0] counter; // Enough for counting up to 32

    // State and counter updates
    always @(posedge clk or negedge reset_n) begin
        if (!reset_n) begin
            state <= RED;
            counter <= 0;
        end else if (enable) begin
            case (state)
                RED: begin
                    if (counter == RED_TIME - 1) begin
                        state <= GREEN;
                        counter <= 0;
                    end else
                        counter <= counter + 1;
                end

                GREEN: begin
                    if (counter == GREEN_TIME - 1) begin
                        state <= YELLOW;
                        counter <= 0;
                    end else
                        counter <= counter + 1;
                end

                YELLOW: begin
                    if (counter == YELLOW_TIME - 1) begin
                        state <= RED;
                        counter <= 0;
                    end else
                        counter <= counter + 1;
                end

                default: begin
                    state <= RED;
                    counter <= 0;
                end
            endcase
        end
    end

    // Output logic
    always @* begin
        // Default outputs
        red = 0;
        yellow = 0;
        green = 0;
        state_code = state;

        case (state)
            RED:    red = 1;
            GREEN:  green = 1;
            YELLOW: yellow = 1;
        endcase
    end

endmodule
'''
filename = "traffic_light/traffic_light.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [20]:
!cd traffic_light/ && iverilog -g2012 -o traffic_light.vvp traffic_light.v traffic_light_tb.v && vvp traffic_light.vvp

All test cases passed!
